# SegOCR — Colab Long Run (multi-account-ready)

A scaled-up training run with **Google Drive checkpointing** so disconnects don't lose progress, plus the targeted tweaks identified by the j-IoU=0.024 diagnosis.

## Single account or multi-account?

This notebook supports both modes via a single knob, `WORKER_ID`, set in the cell after the Drive mount:

- **Single account** (default): keep `WORKER_ID = 0`, run all cells, get one model.
- **Multi-account ensemble**: run this notebook on N different accounts, set `WORKER_ID = 0` on the first, `1` on the second, etc. Each account uses *its own* Drive and produces a model trained on a *distinct* slice of seed-space. After all finish, run `scripts/average_runs.py` on the resulting `averaged_best.pth` files (collected from each account's Drive) to produce the ensemble. Expected gain over a single account: roughly +5-8% absolute fg_miou.

## What's different from the previous experiment

Diagnosis showed the lowercase `j` failure (IoU=0.024) was **not a bug** — it was a resolution problem. At the experiment's smallest font size (22px), `j` differs from `i` by only **4 pixels** of descender hook, easily lost after compositing + degradation. This run targets that:

| Setting | Previous experiment | Long run |
|---|---|---|
| Image size | 256x256 | **384x384** (2.25x pixels) |
| Min font size | 22 px | **32 px** (descender now ~8 px) |
| Samples per worker | 20,000 | **50,000** |
| Iterations | 25,000 | **60,000** |
| `rare_char_boost` | 2.0 | **4.0** |
| Case dist (lower / upper / mixed / title) | 0.35/0.25/0.30/0.10 | **0.50/0.20/0.20/0.10** |
| Drive checkpointing | none | **dataset + every checkpoint persisted** |

**Setup**: `Runtime -> Change runtime type -> T4 GPU` (free) or A100 (Pro).

**Wall time on T4**: ~30 min generation + ~10-12 hours training. **You will hit the free-tier 12-hour limit at least once.** That's fine - the next session resumes from the last checkpoint.

**Wall time on A100**: ~10 min generation + ~3-4 hours training, single session.

## 1 / Mount Google Drive

Drive holds the dataset and checkpoints. If you skip this, everything is ephemeral — you'll lose progress on disconnect.

**Multi-account hint**: if all accounts share access to a Drive Shared Drive, point `DRIVE_ROOT` at the shared path. Each worker writes to its own subdir, and the ensemble step runs in any account without manual file collection.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Single-account: keep this at MyDrive.
# Multi-account on a Shared Drive: change to e.g. '/content/drive/Shareddrives/segocr/longrun'.
DRIVE_ROOT = '/content/drive/MyDrive/segocr_longrun'
os.makedirs(DRIVE_ROOT, exist_ok=True)
DATA_DIR = f'{DRIVE_ROOT}/dataset'
WEIGHTS_DIR = f'{DRIVE_ROOT}/weights'
print(f'Drive root: {DRIVE_ROOT}')
print(f'Dataset:    {DATA_DIR}')
print(f'Weights:    {WEIGHTS_DIR}')

## 2 / Worker config (multi-account knob)

Set `WORKER_ID = 0` for single-account use. For multi-account ensembling, use 0, 1, 2, ... on each respective account. Each value picks a distinct slice of dataset indices and seeds the random generators differently, so the resulting models land in different basins of the loss landscape.

`NUM_WORKERS_TOTAL` should match how many accounts you'll use. Setting it to 1 keeps the single-account path.

In [ ]:
WORKER_ID = 0          # change this on each account: 0, 1, 2, ...
NUM_WORKERS_TOTAL = 1  # set to the total number of accounts you'll use

if NUM_WORKERS_TOTAL > 1:
    WEIGHTS_DIR = f'{DRIVE_ROOT}/weights/worker{WORKER_ID}'
    DATA_DIR = f'{DRIVE_ROOT}/dataset/worker{WORKER_ID}'
    import os
    os.makedirs(WEIGHTS_DIR, exist_ok=True)
    os.makedirs(DATA_DIR, exist_ok=True)
print(f'Worker {WORKER_ID} of {NUM_WORKERS_TOTAL}')
print(f'Dataset: {DATA_DIR}')
print(f'Weights: {WEIGHTS_DIR}')

## 2 · Clone + install

In [ ]:
import os
if not os.path.isdir('/content/segocr'):
    !git clone https://github.com/mauryantitans/SegOCR.git /content/segocr
%cd /content/segocr
!git pull --quiet

In [ ]:
!pip install -q -e .
!pip install -q -r requirements/base.txt
!pip install -q segmentation-models-pytorch wandb

## 3 / Verify GPU + pick a training budget preset

Three presets, each calibrated to a realistic GPU budget. The cell below auto-detects your GPU and picks a default, but you can override `BUDGET` manually.

| Preset | Per-worker target | GPU | Wall time | Notes |
|---|---|---|---|---|
| `free_4hr_ensemble` | 25K samples × 18K iters @ 256² | T4 | ~4hr | Designed for 3-account ensemble (single account is too small alone). |
| `free_12hr` | 50K samples × 60K iters @ 384² | T4 | ~12hr (multi-session) | Single-account long run — needs Drive resume across sessions. |
| `pro_a100` | 100K samples × 80K iters @ 512² | A100 | ~6hr | Paid runtimes — paper-grade single run. |


In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':
    name = torch.cuda.get_device_name(0)
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {name} ({mem_gb:.1f} GB)')
else:
    name, mem_gb = 'cpu', 0
    print('cpu — switch to a GPU runtime before continuing.')

# Auto-pick a preset; override below if you want.
if mem_gb > 30:
    BUDGET = 'pro_a100'
elif NUM_WORKERS_TOTAL > 1:
    BUDGET = 'free_4hr_ensemble'
else:
    BUDGET = 'free_12hr'

# Override here if you want a different preset:
# BUDGET = 'free_4hr_ensemble'  # tight 4hr per worker, designed for 3-account ensemble
# BUDGET = 'free_12hr'           # single account, long run with multi-session resume
# BUDGET = 'pro_a100'            # A100, single-session paper-grade run

PRESETS = {
    'free_4hr_ensemble': dict(
        IMG_SIZE=512, MIN_FONT=40, MAX_FONT=128,
        NUM_IMAGES=12_000, TOTAL_ITERS=10_000,
        BATCH_SIZE=16, ENCODER='resnet18', WARMUP=500,
        EVAL_INTERVAL=1_500, SAVE_INTERVAL=1_500,
    ),
    'free_12hr': dict(
        IMG_SIZE=384, MIN_FONT=32, MAX_FONT=96,
        NUM_IMAGES=50_000, TOTAL_ITERS=60_000,
        BATCH_SIZE=12, ENCODER='resnet34', WARMUP=1_000,
        EVAL_INTERVAL=2_500, SAVE_INTERVAL=2_500,
    ),
    'pro_a100': dict(
        IMG_SIZE=512, MIN_FONT=32, MAX_FONT=128,
        NUM_IMAGES=100_000, TOTAL_ITERS=80_000,
        BATCH_SIZE=24, ENCODER='resnet50', WARMUP=1_500,
        EVAL_INTERVAL=2_500, SAVE_INTERVAL=5_000,
    ),
}
P = PRESETS[BUDGET]
print(f'\nBudget preset: {BUDGET}')
for k, v in P.items():
    print(f'  {k}: {v}')

## 4 / Build the config from the chosen preset

Everything below — image size, model size, training schedule — is driven by the preset selected above. The data-side tweaks identified by the j-IoU diagnosis are baked into all presets:

- Lowercase-biased case distribution (50/20/20/10)
- `rare_char_boost = 4.0`
- Layout weighted toward horizontal/paragraph for clean recognition


In [ ]:
import yaml
from pathlib import Path

NUM_IMAGES = P['NUM_IMAGES']
TOTAL_ITERS = P['TOTAL_ITERS']
IMG_SIZE = P['IMG_SIZE']

cfg = yaml.safe_load(Path('segocr/configs/default.yaml').read_text())

# Generator — fonts
cfg['generator']['fonts']['root_dir'] = '/usr/share/fonts'
cfg['generator']['fonts']['cache_path'] = f'{DRIVE_ROOT}/font_cache_{WORKER_ID}.json'
cfg['generator']['fonts']['min_size'] = P['MIN_FONT']
cfg['generator']['fonts']['max_size'] = P['MAX_FONT']
cfg['generator']['image_size'] = [IMG_SIZE, IMG_SIZE]
cfg['generator']['num_workers'] = 2
cfg['generator']['output_dir'] = DATA_DIR

# Generator — text: stick to short content (max 2-3 words, no paragraphs)
cfg['generator']['text']['min_length'] = 2
cfg['generator']['text']['max_length'] = 20
cfg['generator']['text']['min_words_per_line'] = 1
cfg['generator']['text']['max_words_per_line'] = 3
cfg['generator']['text']['max_lines'] = 1
cfg['generator']['text']['case_distribution'] = {
    'lower': 0.50, 'upper': 0.20, 'mixed': 0.20, 'title': 0.10,
}
cfg['generator']['text']['rare_char_boost'] = 4.0
cfg['generator']['text']['corpus_paths'] = [
    {'path': 'BUNDLED:signs', 'tag': 'signs', 'weight': 0.30},
    {'path': 'BUNDLED:receipts', 'tag': 'receipts', 'weight': 0.20},
    {'path': 'BUNDLED:names', 'tag': 'names', 'weight': 0.20},
    {'path': 'BUNDLED:numbers', 'tag': 'numbers', 'weight': 0.30},
]

# Generator — layout: drop paragraph mode entirely (user observation:
# multi-line / long text gets scaled down to fit canvas, defeating
# min_font_size). Bias toward horizontal/rotated for clean recognition.
cfg['generator']['layout']['modes'] = {
    'horizontal': 0.50, 'rotated': 0.20, 'curved': 0.10,
    'perspective': 0.10, 'deformed': 0.10, 'paragraph': 0.0,
}

# Generator — backgrounds: less procedural noise (was confusing even
# for humans), more solid/gradient.
cfg['generator']['background']['natural_image_dirs'] = []
cfg['generator']['background']['tier_distribution'] = {
    'tier1_solid': 0.40, 'tier2_procedural': 0.30,
    'tier3_natural': 0.25, 'tier4_adversarial': 0.05,
}

# Generator — compositing: bias toward contrast-aware so text is
# legible against busier backgrounds.
cfg['generator']['compositing']['color_strategy'] = {
    'contrast_aware': 0.60, 'random': 0.30, 'low_contrast': 0.10,
}

# Generator — degradation: dial back blur (the mask-aware blur in
# DegradationPipeline.apply_with_mask now propagates blur to the
# class mask via dilation, but we still keep blur conservative).
cfg['generator']['degradation']['blur'] = {
    'probability': 0.30, 'gaussian_sigma': [0.3, 1.0],
    'motion_kernel': [3, 7], 'defocus_radius': [1, 3],
}
cfg['generator']['degradation']['noise']['probability'] = 0.40
cfg['generator']['degradation']['noise']['gaussian_sigma'] = [5, 20]
cfg['generator']['degradation']['occlusion']['probability'] = 0.05

# Model
cfg['model']['architecture'] = 'unet'
cfg['model']['encoder'] = P['ENCODER']
cfg['model']['encoder_weights'] = 'imagenet'
cfg['model']['head_features'] = 64
cfg['model']['decoder_channels'] = [256, 128, 64, 32, 32]
cfg['model']['heads'] = {'semantic': True, 'affinity': True, 'direction': True}
cfg['model']['num_classes'] = 63
cfg['model']['input_size'] = [IMG_SIZE, IMG_SIZE]

# Training
cfg['training']['learning_rate'] = 3e-4
cfg['training']['weight_decay'] = 1e-4
cfg['training']['total_iters'] = TOTAL_ITERS
cfg['training']['warmup_iters'] = P['WARMUP']
cfg['training']['eval_interval'] = P['EVAL_INTERVAL']
cfg['training']['save_interval'] = P['SAVE_INTERVAL']
cfg['training']['log_interval'] = 100
cfg['training']['batch_size'] = P['BATCH_SIZE']
cfg['training']['num_workers'] = 2
cfg['training']['mixed_precision'] = True
cfg['training']['output_dir'] = WEIGHTS_DIR
cfg['training']['ema'] = {'enabled': True, 'decay': 0.999}
cfg['training']['checkpoint_averaging'] = {'enabled': True, 'top_n': 3}
cfg['training']['keep_best_n'] = 5
cfg['training']['wandb'] = {'project': None}

config_path = f'{DRIVE_ROOT}/longrun_config_worker{WORKER_ID}.yaml'
Path(config_path).write_text(yaml.safe_dump(cfg))
print(f'Worker {WORKER_ID}: {NUM_IMAGES} samples × {TOTAL_ITERS} iters @ batch={P["BATCH_SIZE"]}, {IMG_SIZE}px, encoder={P["ENCODER"]}')
print(f'min_font={P["MIN_FONT"]}px, max words=3, no paragraphs')
print(f'Config: {config_path}')

## 5 · Generate the dataset (~30 min on T4, ~15 min on A100)

Skipped automatically if the Drive dataset already has the target sample count — you can keep adding sessions without redoing this step.

In [ ]:
import os
INDEX_OFFSET = WORKER_ID * NUM_IMAGES
n_existing = len(os.listdir(f'{DATA_DIR}/images')) if os.path.isdir(f'{DATA_DIR}/images') else 0
if n_existing >= NUM_IMAGES:
    print(f'Dataset already has {n_existing} samples >= {NUM_IMAGES} target - skipping regeneration.')
else:
    print(f'Existing samples: {n_existing}, target: {NUM_IMAGES} (index offset {INDEX_OFFSET}) - generating...')
    !python -m scripts.generate_dataset --config {config_path} --num-images {NUM_IMAGES} --output {DATA_DIR} --index-offset {INDEX_OFFSET}
    print(f'Now have ' + str(len(os.listdir(f'{DATA_DIR}/images'))) + ' samples.')

## 6 · Quick visual check

8 random samples, with character text overlaid in the title. Verify lowercase letters now appear at decent size and 'j' has a visible descender.

In [ ]:
import json, cv2, random
import matplotlib.pyplot as plt
import numpy as np

random.seed(7)
indices = random.sample(range(INDEX_OFFSET, INDEX_OFFSET + NUM_IMAGES), 8)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, i in zip(axes.flat, indices):
    name = f'{i:06d}'
    img = cv2.cvtColor(cv2.imread(f'{DATA_DIR}/images/{name}.png'), cv2.COLOR_BGR2RGB)
    sem = cv2.imread(f'{DATA_DIR}/semantic/{name}.png', cv2.IMREAD_GRAYSCALE)
    overlay = img.copy()
    overlay[sem > 0] = [255, 0, 0]
    blended = cv2.addWeighted(img, 0.55, overlay, 0.45, 0)
    meta = json.loads(open(f'{DATA_DIR}/metadata/{name}.json').read())
    text = ''.join(c['char'] for c in meta['characters'])[:30]
    ax.imshow(blended)
    ax.set_title(f"[{meta['layout_mode']}] {text}", fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 7 · Train (auto-resumes from latest Drive checkpoint)

**This cell is the long step.** Plan for it to run for hours.

If your runtime disconnects, **just re-run this cell** — `--resume-latest` finds the highest-iteration checkpoint in `WEIGHTS_DIR` and continues from there. The model state, EMA weights, and optimizer state all resume; the LR scheduler fast-forwards to the right step.

Tip while it runs: keep this Colab tab visible. Idle background tabs get throttled by the browser, which can stretch the total wall time. If you have to close the laptop, leave it plugged in — Colab disconnects on extended sleep.

In [ ]:
TRAIN_SEED = WORKER_ID + 1   # 1, 2, 3, ... - different init basin per worker
# Set REPRODUCIBLE = True only if you specifically need bit-exact GPU determinism
# (10-20%% slower).  Most runs don't need this.
REPRODUCIBLE = False

REPRO_FLAG = '--reproducible' if REPRODUCIBLE else ''
!python -m scripts.train_model \
    --config {config_path} \
    --resume-latest {WEIGHTS_DIR} \
    --seed {TRAIN_SEED} \
    {REPRO_FLAG}

## 8 · Load the best checkpoint

In [ ]:
from segocr.models.unet import build_model
from segocr.utils.config import load_config

cfg = load_config(config_path)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_model(cfg['model']).to(device).eval()

ckpts = sorted(Path(WEIGHTS_DIR).glob('checkpoint_*.pth'))
averaged = Path(WEIGHTS_DIR) / 'averaged_best.pth'
if averaged.exists():
    print(f'Loading averaged best: {averaged}')
    state = torch.load(averaged, map_location=device)
elif ckpts:
    print(f'Loading latest: {ckpts[-1]}')
    state = torch.load(ckpts[-1], map_location=device)
else:
    raise FileNotFoundError(f'No checkpoints in {WEIGHTS_DIR}')

if 'ema' in state:
    print('→ Using EMA weights')
    model.load_state_dict(state['ema'], strict=False)
else:
    model.load_state_dict(state['model'])
model.eval()
print(f'Model: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M params')

## 9 · Final validation metrics + targeted j-vs-i diagnostic

We compare the new run's `j` IoU directly against the previous run's 0.024. Goal: significantly above 0.30 — that means the resolution + duration tweaks fixed it.

We also print a small confusion matrix among `j i l 1 I L` to confirm there's no leftover collapse.

In [ ]:
from torch.utils.data import DataLoader
from segocr.training.dataset import SegOCRDataset, collate_fn
from segocr.training.evaluator import Evaluator
from segocr.utils.charset import char_to_class_id, class_id_to_char

val_ds = SegOCRDataset(DATA_DIR, split='val', train_aug=False)
val_loader = DataLoader(
    val_ds, batch_size=cfg['training']['batch_size'], shuffle=False,
    num_workers=2, collate_fn=collate_fn,
)
evaluator = Evaluator(num_classes=cfg['model']['num_classes'], device=device)
metrics = evaluator.evaluate(model, val_loader)

print('───────────────────────────────────────────')
print(f"miou         {metrics['miou']:.3f}")
print(f"fg_miou      {metrics['fg_miou']:.3f}    (foreground only — the OCR signal)")
print(f"binary_miou  {metrics['binary_miou']:.3f}    (text-vs-background)")
print()

char_to_cls = char_to_class_id(tier=1)
diagnostic_chars = ['j', 'i', 'l', '1', 'I', 'L', 'J']
print('IoU for chars that previously confused the model:')
for c in diagnostic_chars:
    cid = char_to_cls[c]
    iou = metrics.get(f'iou_class_{cid:02d}', 0.0)
    flag = '✅' if iou > 0.30 else '⚠️ ' if iou > 0.10 else '❌'
    print(f'  {flag} {c!r}  (class {cid:2d})  IoU={iou:.3f}')
print()

# Mini-confusion matrix among the diagnostic chars + background
diag_class_ids = [0] + [char_to_cls[c] for c in diagnostic_chars]
cm = evaluator.confusion_matrix.cpu().numpy()
sub_cm = cm[np.ix_(diag_class_ids, diag_class_ids)]
header = ['bg'] + diagnostic_chars
print('Confusion matrix (rows = target, cols = prediction):')
print(f"{'':<6}" + ''.join(f'{h:>8}' for h in header))
for i, target_char in enumerate(header):
    row_total = max(1, sub_cm[i].sum())
    pcts = sub_cm[i] / row_total
    print(f'{target_char:<6}' + ''.join(f'{v:>7.1%}' for v in pcts))

## 10 · Per-class IoU breakdown

In [ ]:
import string
id_to_char = class_id_to_char(tier=1)
rows = []
for cid in range(1, cfg['model']['num_classes']):
    iou = metrics.get(f'iou_class_{cid:02d}', 0.0)
    rows.append((id_to_char.get(cid, '?'), cid, iou))
rows_sorted = sorted(rows, key=lambda r: -r[2])
print('Top 15 best-recognized characters:')
for char, cid, iou in rows_sorted[:15]:
    bar = '█' * int(iou * 20)
    print(f'  {char}  (class {cid:2d})  {iou:.3f}  {bar}')
print()
print('Bottom 15 worst-recognized characters:')
for char, cid, iou in rows_sorted[-15:]:
    bar = '█' * int(iou * 20)
    print(f'  {char}  (class {cid:2d})  {iou:.3f}  {bar}')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10))
for ax, (label, group) in zip(axes, [
    ('Uppercase A–Z', string.ascii_uppercase),
    ('Lowercase a–z', string.ascii_lowercase),
    ('Digits 0–9', string.digits),
]):
    chars = list(group)
    ious = []
    for c in chars:
        cid = next((cid for ch, cid, _ in rows if ch == c), None)
        ious.append(metrics.get(f'iou_class_{cid:02d}', 0.0) if cid is not None else 0)
    ax.bar(chars, ious, color=['#2ecc71' if i > 0.3 else '#e74c3c' for i in ious])
    ax.set_ylim(0, 1)
    ax.set_title(f'{label} — per-class IoU')
    ax.axhline(0.3, color='gray', linestyle='--', alpha=0.5)
    ax.set_ylabel('IoU')
plt.tight_layout()
plt.show()

## 11 · Visualize predictions on validation samples

In [ ]:
def colorize_mask(mask, num_classes):
    np.random.seed(42)
    palette = np.random.randint(50, 256, size=(num_classes, 3), dtype=np.uint8)
    palette[0] = (0, 0, 0)
    return palette[mask]

n_show = 6
fig, axes = plt.subplots(n_show, 3, figsize=(12, 4 * n_show))
with torch.no_grad():
    for row in range(n_show):
        sample = val_ds[row]
        x = sample['image'].unsqueeze(0).to(device)
        out = model(x)
        pred = out['semantic'].argmax(dim=1).cpu().numpy()[0]
        gt = sample['targets']['semantic'].cpu().numpy()
        img = sample['image'].permute(1, 2, 0).cpu().numpy()
        img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
        img = (np.clip(img, 0, 1) * 255).astype(np.uint8)
        axes[row, 0].imshow(img); axes[row, 0].set_title('Input', fontsize=10); axes[row, 0].axis('off')
        axes[row, 1].imshow(colorize_mask(gt, cfg['model']['num_classes']))
        axes[row, 1].set_title(f'Ground truth ({(gt > 0).sum()} fg)', fontsize=10); axes[row, 1].axis('off')
        axes[row, 2].imshow(colorize_mask(pred, cfg['model']['num_classes']))
        axes[row, 2].set_title(f'Prediction ({(pred > 0).sum()} fg)', fontsize=10); axes[row, 2].axis('off')
plt.tight_layout()
plt.show()

## 12 · Decode predicted text (left-to-right reading order)

In [ ]:
from scipy.ndimage import label as connected_components

def decode_predicted_text(pred_mask, min_area=10):
    fg = (pred_mask > 0).astype(np.uint8)
    labeled, num = connected_components(fg)
    components = []
    for inst_id in range(1, num + 1):
        ys, xs = np.where(labeled == inst_id)
        if len(xs) < min_area:
            continue
        ids, counts = np.unique(pred_mask[ys, xs], return_counts=True)
        ids_nonzero = ids[ids > 0]
        counts_nonzero = counts[ids > 0]
        if len(ids_nonzero) == 0:
            continue
        top_class = ids_nonzero[counts_nonzero.argmax()]
        components.append((xs.mean(), top_class))
    components.sort(key=lambda c: c[0])
    return ''.join(id_to_char.get(int(cls), '?') for _, cls in components)

horizontal_indices = []
for i, p in enumerate(val_ds.image_paths[:300]):
    meta = json.loads((Path(DATA_DIR) / 'metadata' / f'{p.stem}.json').read_text())
    if meta.get('layout_mode') == 'horizontal':
        horizontal_indices.append(i)
    if len(horizontal_indices) >= 12:
        break

print(f'{"Sample":<10} {"Ground truth":<32} → {"Predicted":<32} match?')
print('-' * 95)
with torch.no_grad():
    for i in horizontal_indices[:12]:
        sample = val_ds[i]
        x = sample['image'].unsqueeze(0).to(device)
        out = model(x)
        pred = out['semantic'].argmax(dim=1).cpu().numpy()[0]
        meta = json.loads((Path(DATA_DIR) / 'metadata' / f'{val_ds.image_paths[i].stem}.json').read_text())
        gt_text = ''.join(c['char'] for c in meta['characters'])
        pred_text = decode_predicted_text(pred)
        match = sum(1 for c in pred_text if c in gt_text) / max(1, len(gt_text))
        print(f'{val_ds.image_paths[i].stem:<10} {gt_text:<32} → {pred_text:<32} {match:.0%}')

## What to look for

Compared to the previous experiment (binary_miou=0.73, fg_miou=0.36, j IoU=0.024), this run targets:

- **`binary_miou` > 0.92** — more iterations + larger images = sharper text/bg boundary.
- **`fg_miou` > 0.55** — character classification gets the bulk of the gain.
- **`j` IoU > 0.30** — the headline diagnostic. If j is now green, the resolution fix worked.
- **Lowercase IoU bars mostly green** — case-distribution rebalance addresses the uppercase corpus bias.
- **Decoded text > 70% match for clean horizontal samples** — the model commits to letters confidently.

If `j` IoU is still red, the next step is to check the confusion matrix above — look at the `j` row to see what the model is predicting `j` pixels as. If most go to `i`, you need *more* `j` exposure (set `rare_char_boost: 8.0` and a lowercase-letters-only corpus). If they go to background, the model is giving up — train longer.

## To go further from here

- **Real backgrounds**: `bash scripts/setup_data.sh` for COCO + DTD (~25 GB) → enables Tier 3 (50%) instead of falling back to procedural.
- **SegFormer encoder**: install `mmsegmentation` and switch `model.architecture: segformer`. Higher capacity ceiling than UNet.
- **Real-image evaluation**: download ICDAR/COCO-Text/Total-Text and compare to Tesseract / PaddleOCR / TrOCR baselines (Phase 2 evaluation in the Implementation Guide).

## 13 / Build the ensemble (multi-account workflow only)

Once all N accounts have finished training, you have N copies of `averaged_best.pth` - one in each account's Drive. Two ways to build the ensemble:

**Option A: collect locally**:

1. Download `<DRIVE_ROOT>/weights/worker<N>/averaged_best.pth` from each account.
2. On your laptop (or any machine with the repo):
   ```bash
   python -m scripts.average_runs \
       --checkpoints worker0/averaged_best.pth worker1/averaged_best.pth worker2/averaged_best.pth \
       --output ensemble.pth
   ```

**Option B: pool checkpoints into one Drive, average in Colab**:

1. Each worker uploads `averaged_best.pth` to a shared Drive folder (or Drive Shared Drives, or manual upload).
2. Run the cell below in any account that has access to the pooled folder.

**Evaluating the ensemble**: load `ensemble.pth` via the same path as cell 8; it has both `model` and `ema` keys set to the averaged state. Re-run cells 9-12 with WEIGHTS_DIR pointing at the ensemble file to see metrics + per-class IoU + decoded text. Expect binary_miou and fg_miou to land a few percent above any single worker's numbers.

In [ ]:
# Option B: average pooled worker outputs in-Colab.
# Edit ENSEMBLE_INPUTS to match your actual collected checkpoint paths.
ENSEMBLE_INPUTS = [
    f'{DRIVE_ROOT}/weights/worker0/averaged_best.pth',
    f'{DRIVE_ROOT}/weights/worker1/averaged_best.pth',
    f'{DRIVE_ROOT}/weights/worker2/averaged_best.pth',
]
ENSEMBLE_OUTPUT = f'{DRIVE_ROOT}/weights/ensemble.pth'

from pathlib import Path
if all(Path(p).exists() for p in ENSEMBLE_INPUTS):
    !python -m scripts.average_runs \
        --checkpoints {' '.join(ENSEMBLE_INPUTS)} \
        --output {ENSEMBLE_OUTPUT}
    print(f'\nEnsemble written to {ENSEMBLE_OUTPUT}.')
    print('Re-run cells 8-12 with the load path swapped to ensemble.pth to evaluate.')
else:
    missing = [p for p in ENSEMBLE_INPUTS if not Path(p).exists()]
    print(f'Missing inputs (skip if single-account): {missing}')